In [ ]:
# Analysis & Prediction - All in One Cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
import os

# --- Путь к файлам ---
data_path = 'data/aapl.csv'
models_path = 'models/'
output_path = 'data/aapl_predicted.csv'

# --- Загрузка данных ---
df = pd.read_csv(data_path)
prices = df['Actual'].values.reshape(-1, 1)

# --- Масштабирование ---
scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices)

# --- Подготовка последовательностей ---
seq_length = 9
X = []
for i in range(len(prices_scaled) - seq_length):
    X.append(prices_scaled[i:i+seq_length])
X = np.array(X)

# --- Загрузка моделей ---
lstm_model = load_model(os.path.join(models_path, 'lstm_model.h5'))
transformer_model = load_model(os.path.join(models_path, 'transformer_model.keras'))

# --- Предсказания ---
lstm_pred = lstm_model.predict(X)
transformer_pred = transformer_model.predict(X)

# --- Обратное масштабирование ---
lstm_pred_rescaled = scaler.inverse_transform(lstm_pred)
transformer_pred_rescaled = scaler.inverse_transform(transformer_pred)
y_actual = prices[seq_length:]

# --- Сохранение результатов ---
result_df = pd.DataFrame({
    'Actual': y_actual.flatten(),
    'LSTM_Predicted': lstm_pred_rescaled.flatten(),
    'Transformer_Predicted': transformer_pred_rescaled.flatten()
})
os.makedirs('data', exist_ok=True)
result_df.to_csv(output_path, index=False)
print(f'Предсказания сохранены в {output_path}')

# --- Визуализация ---
plt.figure(figsize=(12,6))
plt.plot(result_df['Actual'], label='Actual', marker='o')
plt.plot(result_df['LSTM_Predicted'], label='LSTM Predicted', linestyle='--')
plt.plot(result_df['Transformer_Predicted'], label='Transformer Predicted', linestyle=':')
plt.title('AAPL Stock Prediction')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show()